# L3 — Le streaming

<img src="./assets/LC_streaming.png" width="400">

Le **streaming** réduit la latence perçue par l'utilisateur en affichant la réponse
au fur et à mesure qu'elle est générée, plutôt que d'attendre la réponse complète.

## Objectifs pédagogiques

À la fin de cette leçon, vous serez capable de :

- Comprendre la différence entre `invoke` et `stream`
- Utiliser les trois modes de streaming de LangChain : `values`, `messages`, `custom`
- Streamer des données depuis un outil
- Choisir le bon mode selon le cas d'usage

---

> **Rappel L2 :** Nous avons vu que LangChain représente une conversation comme une liste de messages.
> Dans ce notebook, nous allons voir comment ces messages peuvent **arriver progressivement**.

## 1. invoke vs stream — La différence fondamentale

```text
invoke

Question ────────────────────────────────→ Réponse complète
         (on attend... on attend... on attend)


stream

Question → morceau → morceau → morceau → ... → fin
           (l'utilisateur voit la réponse se construire)
```

Le contenu **final est identique**. La différence est l'**expérience utilisateur** :
avec `stream`, l'utilisateur voit la réponse se construire en temps réel.

## 2. Préparer l'environnement

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Charge les variables d'environnement depuis le fichier .env
load_dotenv()

# Vérification de la présence de la clé API Mistral
if not os.getenv("MISTRAL_API_KEY"):
    raise ValueError("❌ MISTRAL_API_KEY non trouvée. Créez un fichier .env avec MISTRAL_API_KEY=votre_clé")
if not os.getenv("SERVER_URL"):
    raise ValueError("❌ SERVER_URL non trouvée. Créez un fichier .env avec SERVER_URL=votre_url")

print("✅ Clé API Mistral chargée.")
print("✅ URL du serveur Mistral chargée.")

# Vérification des variables d'environnement et des packages requis
from env_utils import doublecheck_env
doublecheck_env("example.env")  # vérification des variables de l'environment

✅ Clé API Mistral chargée.
✅ URL du serveur Mistral chargée.
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.



In [5]:
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent

# Nom du modèle — une seule constante pour garder la cohérence entre les notebooks.
# Note : le serveur dédié n'expose PAS mistral-large-latest.
# mistral-medium-latest est le modèle de génération le plus capable disponible ici.
MODEL = "mistral-medium-latest"

# Création du modèle Mistral que LangChain utilisera pour générer les réponses.
# temperature=0 garantit des réponses déterministes (utile pour le SQL).
# base_url (alias de endpoint) pointe vers le serveur dédié — SERVER_URL inclut /v1.
llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    api_key=os.getenv("MISTRAL_API_KEY"),
    base_url=os.getenv("SERVER_URL"),
)

print(f"✅ Modèle {MODEL} initialisé.")

agent = create_agent(
    model=llm,
    system_prompt="Tu es un humoriste spécialiste du développement logiciel. Tu réponds en français.",
)

print(f"✅ Agent avec {MODEL} initialisé.")

✅ Modèle mistral-medium-latest initialisé.
✅ Agent avec mistral-medium-latest initialisé.


## 3. Sans streaming — `invoke`

### Objectif

Observer le comportement sans streaming : on attend la réponse complète.

### Méthode

`agent.invoke()` envoie la question au modèle et **attend** la réponse complète
avant de retourner quoi que ce soit.

### Résultat attendu

La réponse s'affiche **d'un seul coup** après un délai d'attente.

In [6]:
# invoke() attend la réponse complète avant de retourner.
# Toute la latence est concentrée ici — on ne voit rien jusqu'à la fin.
result = agent.invoke({"messages": [{"role": "user", "content": "Raconte-moi une blague"}]})
print(result["messages"][1].content)

Bien sûr ! En voici une pour les développeurs :

**Pourquoi les développeurs confondent-ils toujours Halloween et Noël ?**
*Parce que OCT 31 == DEC 25 !* 🎃🎄

*(Explication pour les non-inités : en octal (base 8), 31 = 25 en décimal (base 10). Oui, on est des gens drôles.)*

Tu veux une autre ? 😄


## 4. Streaming — mode `values`

### Objectif

Observer le streaming en mode `values` : l'état complet après chaque étape.

### Méthode

`agent.stream()` avec `stream_mode="values"` retourne **l'état complet** des messages
après chaque étape de l'agent (question reçue → décision → outil → réponse).

C'est le mode utilisé dans **L1** pour suivre les étapes de l'agent SQL.

### Résultat attendu

Vous verrez apparaître les messages un par un : d'abord la question, puis la réponse.

In [7]:
# stream_mode="values" retourne l'état complet à chaque étape de l'agent.
# C'est utile pour voir la progression de la conversation.
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Raconte-moi une blague de développeur"}]},
    stream_mode="values",
):
    # À chaque étape, on affiche uniquement le dernier message
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Raconte-moi une blague de développeur
================================== Ai Message ==================================

Bien sûr ! En voici une qui devrait te faire sourire (ou grogner, selon ton niveau de caféine) :

**Pourquoi les développeurs confondent-ils toujours Halloween et Noël ?**
*Parce que OCT 31 == DEC 25 !*

*(Explication pour les non-initiés : en octal (base 8), 31 s'écrit "31", et en décimal (base 10), 25 s'écrit... "25". Mais en hexadécimal, 31 en décimal vaut 1F, et 25 en décimal vaut 19... Bon, ok, la blague est peut-être un peu trop "niche".)*

Et une autre pour la route :
**Un développeur entre dans un bar, commande une bière, puis une autre, puis une autre...**
Le barman lui dit : *"Tu devrais arrêter de boire, tu as l’air saoul !"*
Le développeur répond : *"Non, non, je fais juste du *recursive drinking*."*

*(Et oui, il a oublié la condition de sortie de la boucle...)*

Tu en veux 

## 5. Streaming — mode `messages` (token par token)

### Objectif

Afficher la réponse **mot par mot**, comme dans une interface de chat.

### Méthode

`stream_mode="messages"` retourne les tokens **un par un** au fur et à mesure
qu'ils sont générés par le modèle. C'est la plus faible latence possible.

```text
stream_mode="messages"

  "Je"  →  " suis"  →  " un"  →  " comédien"  →  "..."
```

### Résultat attendu

Vous verrez la réponse s'afficher progressivement, comme dans ChatGPT.

In [14]:
# stream_mode="messages" retourne les tokens un par un.
# end="" supprime le retour à la ligne entre chaque token.
# flush=True force l'affichage immédiat du token.
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Écris-moi un poème court sur les bugs informatiques."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")
    # print(f"{token.content}", end="", flush=True)

**"Ode au Bug"**

Ô toi, bug malicieux,
Qui dans mon code es si heureux,
Tu danses entre les lignes,
Et fais planter mes cygnes.

Je te cherche, je te traque,
Mais tu ris, coquin, en cache.
Un point-virgule oublié,
Et tout est à recommencer !

*Moralité : Le bug est un art,
Surtout quand il part... en prod.* 😄

## 6. Streaming depuis les outils

### Objectif

Comprendre comment streamer des données directement depuis un outil.

### Méthode

`get_stream_writer()` de LangGraph permet à un outil d'émettre des données
**pendant son exécution**, avant même d'avoir fini.

On utilise ensuite `stream_mode=["values", "custom"]` pour recevoir à la fois
les messages de l'agent et les données custom de l'outil.

### Résultat attendu

Vous verrez les messages de l'outil s'afficher **pendant** que l'outil travaille.

In [9]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_meteo(ville: str) -> str:
    """Obtenir la météo pour une ville donnée."""
    # get_stream_writer() permet d'émettre des données pendant l'exécution de l'outil
    writer = get_stream_writer()
    writer(f"🔍 Recherche des données pour : {ville}")
    writer(f"✅ Données récupérées pour : {ville}")
    # Retourne le résultat (simulé ici)
    return f"Il fait toujours beau à {ville} !"


# Agent avec un outil qui utilise le streaming custom
agent_meteo = create_agent(
    model=llm,
    tools=[get_meteo],
    system_prompt="Tu es un assistant météo. Tu réponds en français.",
)

# stream_mode=["values", "custom"] reçoit les deux flux simultanément
for chunk in agent_meteo.stream(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Paris ?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='Quelle est la météo à Paris ?', additional_kwargs={}, response_metadata={}, id='b8156e9e-b538-4ca6-8751-bc0f8a094b53')]})
('values', {'messages': [HumanMessage(content='Quelle est la météo à Paris ?', additional_kwargs={}, response_metadata={}, id='b8156e9e-b538-4ca6-8751-bc0f8a094b53'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dmdtqx1o9', 'type': 'function', 'function': {'name': 'get_meteo', 'arguments': '{"ville": "Paris"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 103, 'total_tokens': 116, 'completion_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--01a05db1-0604-78c3-81b3-299e66733d89-0', tool_calls=[{'name': 'get_meteo', 'args': {'ville': 'Paris'}, 'id': 'dmdtqx1o9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={

In [10]:
# On peut filtrer pour ne voir que les données custom (de l'outil)
for chunk in agent_meteo.stream(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Lyon ?"}]},
    stream_mode=["custom"],
):
    print(chunk[1])

🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon
🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon
🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon
🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon
🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon
🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


## 7. Résumé des modes de streaming

| Mode | Description | Usage |
|------|-------------|-------|
| `invoke` | Pas de streaming — attend la réponse complète | Scripts, traitements en lot |
| `stream_mode="values"` | État complet après chaque étape | Suivi de la progression de l'agent |
| `stream_mode="messages"` | Tokens un par un | Interfaces de chat, chatbots |
| `stream_mode=["custom"]` | Données émises depuis les outils | Feedback de progression |
| `stream_mode=["values", "custom"]` | Combinaison des deux | Applications complètes |

### 🎯 À vous de jouer !

Modifiez le mode de streaming et observez les différences de comportement.

In [ ]:
# Expérimentez avec différents modes de streaming
for chunk in agent_meteo.stream(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Marseille ?"}]},
    stream_mode=["values", "custom"],  # ← Modifiez ce paramètre
):
    if chunk[0] == "custom":
        print(chunk[1])

## Ce qu'il faut retenir

- `invoke()` attend la réponse **complète** avant de retourner.
- `stream()` avec `stream_mode="values"` retourne l'**état complet** après chaque étape.
- `stream()` avec `stream_mode="messages"` retourne les **tokens un par un**.
- `get_stream_writer()` permet de streamer des données depuis un **outil**.
- Le contenu final est **identique** ; seule l'expérience utilisateur change.

---

> **Prochaine étape →** Dans **L4**, nous allons apprendre à créer des **outils** plus complexes
> et voir comment le modèle Mistral décide quand et comment les utiliser.

## Documentation officielle

- [LangChain — Streaming](https://python.langchain.com/docs/concepts/streaming/)
- [LangGraph — stream_mode](https://langchain-ai.github.io/langgraph/concepts/streaming/)
- [langchain-mistralai — ChatMistralAI](https://python.langchain.com/docs/integrations/chat/mistralai/)
- [Mistral AI — Documentation API](https://docs.mistral.ai/)